In [ ]:
pip install gensim nltk rake-nltk yake keybert PyMuPDF


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz  # PyMuPDF
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter
import nltk

nltk.download('punkt')
nltk.download('stopwords')

In [125]:
def extract_text_pymupdf(pdf_path):
    """Extract text from PDF using PyMuPDF (fitz)."""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text

    except Exception as e:
        print(f"Error using PyMuPDF on {pdf_path}: {e}")
        return None

pdf_path = "/content/drive/MyDrive/Task_pdfs/pdf5.pdf"
extracted_text = extract_text_pymupdf(pdf_path)


In [126]:
extracted_text

'-\n• \n.. \nS.C.R. \nSUPREME COURT REPORTS \n889 \nthe learned English Judges in the first tea case would \nnot be without relevance on the question of sentence \nin many cases of this kind. There can, I think, be no \ndoubt that businessmen who are not lawyers \nmight \nwell be misled into thinking that the Ordinance and \nthe Act did not intend to keep the Order of 1944 alive \nbecause the Order related to certain specified spices \nwhile the Ordinance and the Act changed the .nomen-\nclature and limited themsleves to "foodstuffs", a term \nwhich, on a narrow view, would not include con-\ndiments and spices. \nHowever, these observations are \nnot relevant here because we are not asked to restore \neither the conviction or the sentence. In view of that, \nthere will be no further order and the acquittal \nwill be left as it stands. \n·. \n\\ \nOrder accordingly. \nAgent for the appellant: P. A. Mehta. \nAgent for the respondent : M. S. K. Sastri. \nTHE STATE OF BIHAR \nfl. \n\' \nMA

In [128]:
custom_stop_words = [
    'judgeship', 'court', 'hon’ble', 'proceeding', 'order',
    'judge', 'officer', 'hon', 'ble', 'dated',
    'defendant', 'plaintiff', 'case', 'section', 'law',
    'legal', 'filing', 'shall', 'case', 'cases', 'day', 'act', 'request', 'letter', 'officers',
    'directed', 'district', 'kindly', 'please', 'ensure', 'court',
    'should', 'may', 'must', 'also', 'that', 'is', 'for', 'by', 'on',
    'the', 'and', 'or', 'in', 'with', 'to', 'of', 'as', 'at', 'it',
    'this', 'there', 'these', 'those', 'have', 'has', 'had', 'if',
    'not', 'be', 'we', 'you', 'your', 'their', 'they', 'which',
    'who', 'what', 'when', 'where', 'how', 'more', 'most', 'some',
    'any', 'all', 'few', 'such', 'while', 'between', 'among', 'from',
    'while', 'along', 'other', 'one', 'two', 'three', 'four', 'five','copy','working','concerned','courts','directions','gii','sessions','appeals',
    'work','control', 'aforesaid', 'appeal'

]

#### TF-IDF without custom_stop_words

In [127]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_keywords(extracted_text, num_keywords=10):
    # Create a TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(stop_words='english')

    # Fit and transform the text
    tfidf_matrix = vectorizer.fit_transform([extracted_text])

    # Get feature names (words)
    feature_names = np.array(vectorizer.get_feature_names_out())

    # Get sorted indices in descending order of TF-IDF scores
    sorted_indices = np.argsort(tfidf_matrix.toarray()).flatten()[::-1]

    # Get top keywords
    top_keywords = feature_names[sorted_indices][:num_keywords]

    return top_keywords

# Assuming extracted_text is already defined
keywords = extract_keywords(extracted_text)
print("Top keywords:")
for keyword in keywords:
    print(keyword)

Top keywords:
state
compensation
act
31
entry
article
public
law
acquisition
power


### TF-IDF with custom_stop_words

In [129]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import _stop_words

def extract_keywords(extracted_text, custom_stop_words, num_keywords=10):
    # Combine default English stop words with custom stop words
    all_stop_words = list(_stop_words.ENGLISH_STOP_WORDS) + list(custom_stop_words)

    # Create a TF-IDF Vectorizer with custom stop words
    vectorizer = TfidfVectorizer(stop_words=all_stop_words)

    # Fit and transform the text
    tfidf_matrix = vectorizer.fit_transform([extracted_text])

    # Get feature names (words)
    feature_names = np.array(vectorizer.get_feature_names_out())

    # Get sorted indices in descending order of TF-IDF scores
    sorted_indices = np.argsort(tfidf_matrix.toarray()).flatten()[::-1]

    # Get top keywords
    top_keywords = feature_names[sorted_indices][:num_keywords]

    return top_keywords

# Example usage:
# Assuming extracted_text is already defined
keywords = extract_keywords(extracted_text, custom_stop_words)
print("Top keywords:")
for keyword in keywords:
    print(keyword)

Top keywords:
state
compensation
31
entry
article
public
acquisition
power
purpose
list


### Rake without custom_stop_words

In [130]:
import numpy as np
from rake_nltk import Rake

def extract_keywords_rake(extracted_text, custom_stop_words, num_keywords=10):
    # Combine default English stop words with custom stop words
    all_stop_words = list(custom_stop_words)

    # Initialize RAKE without custom stop words for testing
    rake = Rake()  # Start without custom stop words to debug

    # Extract keywords
    rake.extract_keywords_from_text(extracted_text)

    # Get ranked phrases with their scores
    ranked_phrases = rake.get_ranked_phrases_with_scores()

    # Select the top keywords based on scores
    top_keywords = [phrase for score, phrase in ranked_phrases[:num_keywords]]

    return top_keywords


keywords = extract_keywords_rake(extracted_text, custom_stop_words)
print("Top keywords:")
for keyword in keywords:
    print(keyword)


Top keywords:
4hiraia sir kameshwar singh • f darbhanga • al otherr
known maxim expressum f acit cessare tacit um
dhiraja sir · kame sh war singh
tlhiraja sir kameshwar singh • f darbhanga • nd others
supreme court reports 917 said bihar land reforms act
two cases long island water supply company v
governance kas ~· h ~• f
cumstance unmiistakabry establishes thait entry 36
maharashtra sugar mills ltd .('), also
de jure belli et pacis ''


In [131]:
custom_stop_words = [
    'judgeship', 'court', 'hon’ble', 'proceeding', 'order',
    'judge', 'officer', 'hon', 'ble', 'dated',
    'defendant', 'plaintiff', 'case', 'section', 'law',
    'legal', 'filing', 'shall', 'case', 'cases', 'day', 'act', 'request', 'letter', 'officers',
    'directed', 'district', 'kindly', 'please', 'ensure', 'court',
    'should', 'may', 'must', 'also', 'that', 'is', 'for', 'by', 'on',
    'the', 'and', 'or', 'in', 'with', 'to', 'of', 'as', 'at', 'it',
    'this', 'there', 'these', 'those', 'have', 'has', 'had', 'if',
    'not', 'be', 'we', 'you', 'your', 'their', 'they', 'which',
    'who', 'what', 'when', 'where', 'how', 'more', 'most', 'some',
    'any', 'all', 'few', 'such', 'while', 'between', 'among', 'from',
    'while', 'along', 'other', 'one', 'two', 'three', 'four', 'five','copy','working','concerned','courts','directions','gii','sessions','appeals',
    'work','control', 'aforesaid', 'appeal'

]

### Yake with custom stop words

In [132]:
import string
import yake

def preprocess_text(text, custom_stop_words):
    """
    Preprocess the input text by lowering the case, removing punctuation,
    and filtering out custom stop words.

    Parameters:
    - text (str): The input text to preprocess.
    - custom_stop_words (list): A list of stop words to remove.

    Returns:
    - str: The cleaned text with stop words removed.
    """
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Split the text into words
    text_words = text.split()

    # Create a set of stop words for quick lookup
    stop_words = set(word.lower() for word in custom_stop_words)

    # Filter out stop words and short words (length <= 2)
    filtered_words = [word for word in text_words if word not in stop_words and len(word) > 2]

    # Return the cleaned text
    return ' '.join(filtered_words)

def extract_keywords_yake(extracted_text, custom_stop_words, num_keywords=10):
    """
    Extract keywords from the provided text using YAKE after preprocessing.

    Parameters:
    - extracted_text (str): The text from which to extract keywords.
    - custom_stop_words (list): A list of stop words to remove.
    - num_keywords (int): The number of top keywords to return.

    Returns:
    - list: A list of top keywords extracted from the text.
    """
    # Preprocess the extracted text
    cleaned_text = preprocess_text(extracted_text, custom_stop_words)

    # Create YAKE keyword extractor
    extractor = yake.KeywordExtractor(n=1)
    # Extract keywords
    keywords = extractor.extract_keywords(cleaned_text)

    # Filter out stop words from the extracted keywords
    stop_words_set = set(word.lower() for word in custom_stop_words)
    filtered_keywords = [keyword for keyword, score in keywords if keyword not in stop_words_set]

    # Sort keywords based on score and take the top ones
    sorted_keywords = sorted(filtered_keywords, key=lambda x: x[1])
    top_keywords = [keyword for keyword in sorted_keywords[:num_keywords]]

    return top_keywords



# Extract keywords
keywords = extract_keywords_yake(extracted_text, custom_stop_words)
print("Top keywords:")
for keyword in keywords:
    print(keyword)


Top keywords:
darbhanga
kameshwar
acquisition
legislature
reports
list
bihar
singh
sir
entry


### KeyBert

In [114]:
pip install keybert

In [134]:
from keybert import KeyBERT
import re


# Initialize the KeyBERT model
model = KeyBERT('distilbert-base-nli-mean-tokens')

def remove_integers(text):
    # Use regex to remove any integers
    return re.sub(r'\b\d+\b', '', text)

# Sample extracted text
extracted_text = extracted_text

# Remove integers from the text
cleaned_text = remove_integers(extracted_text)

# Now you can extract keywords without including integers
keywords = model.extract_keywords(cleaned_text, stop_words='english', top_n=100)

# Print the keywords
print("Keywords:")
for keyword in keywords:
    print(keyword)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Keywords:
('tea', 0.3163)
('businessmen', 0.2453)
('spices', 0.2443)
('foodstuffs', 0.2045)
('corporations', 0.202)
('brewer', 0.195)
('overruled', 0.1942)
('disafforestation', 0.1926)
('republican', 0.1901)
('abdicated', 0.1849)
('abolish', 0.1777)
('lawyers', 0.1774)
('economics', 0.1753)
('jurists', 0.1753)
('acquittal', 0.1749)
('oxford', 0.1733)
('impeached', 0.1615)
('overrule', 0.1609)
('farmers', 0.1587)
('acqms1t10n', 0.1529)
('unconstitutional', 0.1516)
('discriminatory', 0.1482)
('attorney', 0.1472)
('brothers', 0.1469)
('masters', 0.1455)
('derogate', 0.1454)
('opera', 0.1434)
('abdication', 0.1434)
('disallowed', 0.1428)
('professor', 0.1427)
('dishonest', 0.1386)
('fraudulent', 0.1383)
('unconstitutionality', 0.1379)
('misapprehension', 0.1373)
('expropriated', 0.1358)
('president', 0.1351)
('old', 0.1345)
('prov1sjon', 0.1339)
('expropriating', 0.1334)
('brewery', 0.1318)
('parliamentary', 0.1317)
('jurisprudence', 0.1315)
('commuted', 0.1309)
('prov1s1ons', 0.1304)
('mo